In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU is available
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# This should show you have a GPU!
if tf.config.list_physical_devices('GPU'):
    print("✓ GPU is enabled! Training will be FAST! 🚀")
else:
    print("⚠️ GPU not enabled. Go to Runtime → Change runtime type → GPU")

In [ ]:
# If you uploaded folders directly
import os

# Point to your Google Drive location
DATA_DIR = './data/chest_xray'
# Verify it exists
if os.path.exists(DATA_DIR):
    print("✓ Dataset found!")
    for split in ['train', 'val', 'test']:
        for category in ['NORMAL', 'PNEUMONIA']:
            path = os.path.join(DATA_DIR, split, category)
            if os.path.exists(path):
                count = len(os.listdir(path))
                print(f"  {split}/{category}: {count} images")
else:
    print(f"❌ Dataset not found at {DATA_DIR}")
    print("Update DATA_DIR to match your Google Drive path")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np
import os
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

class PneumoniaCNNClassifier:
    def __init__(self, input_shape=(256, 256, 1), num_classes=2):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.model = None
        self.history = None

    def build_model(self, architecture='standard'):
        if architecture == 'standard':
            self.model = self._build_standard_cnn()
        else:
            raise ValueError("Architecture must be 'standard'")
        return self.model

    def _build_standard_cnn(self):
        model = models.Sequential([
            layers.Input(shape=self.input_shape),
            layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
            layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
            layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
            layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            layers.Flatten(),
            layers.Dense(256, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(128, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(self.num_classes, activation='softmax')
        ])
        return model

    def compile_model(self, learning_rate=0.001, optimizer='adam'):
        opt = keras.optimizers.Adam(learning_rate=learning_rate) if optimizer == 'adam' else optimizer
        self.model.compile(
            optimizer=opt,
            loss='categorical_crossentropy',
            metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='auc')]
        )
        print("Model compiled successfully!")

    def get_callbacks(self, model_save_path='best_pneumonia_model.keras'):
        return [
            ModelCheckpoint(model_save_path, monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
            EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
        ]

    def train(self, train_generator, validation_generator, epochs=50, callbacks=None):
        if callbacks is None: callbacks = self.get_callbacks()
        self.history = self.model.fit(train_generator, validation_data=validation_generator, epochs=epochs, callbacks=callbacks, verbose=1)
        return self.history

    def plot_training_history(self):
        if not self.history: return
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].plot(self.history.history['accuracy'], label='Train')
        axes[0].plot(self.history.history['val_accuracy'], label='Val')
        axes[0].set_title('Accuracy')
        axes[1].plot(self.history.history['loss'], label='Train')
        axes[1].plot(self.history.history['val_loss'], label='Val')
        axes[1].set_title('Loss')
        plt.show()

    def evaluate(self, test_generator):
        results = self.model.evaluate(test_generator, verbose=1)
        return {'loss': results[0], 'accuracy': results[1], 'precision': results[2], 'recall': results[3], 'auc': results[4]}

    def predict_and_visualize(self, test_generator, num_samples=9):
        predictions = self.model.predict(test_generator, verbose=1)
        predicted_classes = np.argmax(predictions, axis=1)
        true_classes = test_generator.classes
        print(classification_report(true_classes, predicted_classes, target_names=list(test_generator.class_indices.keys())))

print("✓ Optimized CNN Classifier class defined")

In [ ]:
def create_data_generators(data_dir, img_size=(256, 256), batch_size=32, grayscale=True):
    """
    Create train, validation, and test data generators
    """

    color_mode = 'grayscale' if grayscale else 'rgb'

    train_datagen = ImageDataGenerator(
        rescale=1.0/255.0,
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    test_datagen = ImageDataGenerator(rescale=1.0/255.0)

    train_generator = train_datagen.flow_from_directory(
        os.path.join(data_dir, 'train'),
        target_size=img_size,
        batch_size=batch_size,
        class_mode='categorical',
        color_mode=color_mode,
        shuffle=True
    )

    val_generator = test_datagen.flow_from_directory(
        os.path.join(data_dir, 'val'),
        target_size=img_size,
        batch_size=batch_size,
        class_mode='categorical',
        color_mode=color_mode,
        shuffle=False
    )

    test_generator = test_datagen.flow_from_directory(
        os.path.join(data_dir, 'test'),
        target_size=img_size,
        batch_size=batch_size,
        class_mode='categorical',
        color_mode=color_mode,
        shuffle=False
    )

    return train_generator, val_generator, test_generator

print("✓ Data generator function defined")

In [10]:
config_content = """
# config.yaml for Pneumonia Detection Project

# --- Paths Configuration ---
# Base directory for your dataset. Relative paths are recommended.
# Example: './data/chest_xray' if your data is in a 'data' subfolder
data_dir: './data/chest_xray' # Update this path to match your setup.

# Path where the trained model will be saved or loaded from.
# Example: './models/best_pneumonia_cnn.keras' if models are in a 'models' subfolder
model_save_path: './models/best_pneumonia_cnn.keras' # Update this path to match your setup.

# --- Model Training Configuration ---
img_size: [256, 256]
batch_size: 32
epochs: 50
learning_rate: 0.001

# --- Other Configurations (Optional) ---
# For example, if you have different model architectures
model_architecture: 'standard' # 'standard', 'simple'

# Add any other configuration parameters here
"""

with open('config.yaml', 'w') as f:
    f.write(config_content.strip())

print("✓ Created config.yaml template.")
print("Please review and adjust the paths and configurations inside 'config.yaml' as per your project setup.")

✓ Created config.yaml template.
Please review and adjust the paths and configurations inside 'config.yaml' as per your project setup.


In [ ]:
# Configuration (now loaded from config.yaml and environment variables in the previous cell)
# Ensure DATA_DIR, IMG_SIZE, BATCH_SIZE, EPOCHS, LEARNING_RATE, MODEL_SAVE_PATH are defined from config

print("=" * 60)
print("PNEUMONIA DETECTION CNN - TRAINING")
print("=" * 60)

# Verification step to catch the error early
if not os.path.exists(DATA_DIR):
    print(f"❌ ERROR: The directory {DATA_DIR} was not found.")
    print("Please check if Google Drive is mounted and the path is correct, or if your local path exists.")
else:
    # Create data generators
    print("\n1. Creating data generators...")
    train_gen, val_gen, test_gen = create_data_generators(
        DATA_DIR,
        img_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        grayscale=True
    )

    # Initialize classifier
    print("\n2. Building CNN model...")
    classifier = PneumoniaCNNClassifier(
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 1),
        num_classes=len(train_gen.class_indices)
    )

    # Build and compile model
    classifier.build_model(architecture=MODEL_ARCHITECTURE)
    classifier.compile_model(learning_rate=LEARNING_RATE, optimizer='adam')

    # Train model
    print("\n3. Training model...")
    history = classifier.train(
        train_gen,
        val_gen,
        epochs=EPOCHS,
        callbacks=classifier.get_callbacks(MODEL_SAVE_PATH)
    )

    print("\n✓ Training complete!")

In [ ]:
import tensorflow as tf
from tensorflow.keras.metrics import Precision, Recall, AUC

# Mount Google Drive if not already mounted (if MODEL_SAVE_PATH is on Drive)
# Check if DATA_DIR or MODEL_SAVE_PATH are on Google Drive before mounting
if 'drive' in DATA_DIR or 'drive' in MODEL_SAVE_PATH:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print(f"Could not mount Google Drive: {e}")

# Use MODEL_SAVE_PATH from the loaded configuration
# Ensure the configuration is loaded from a previous cell first
# model_path is expected to be defined by a prior configuration loading cell
if 'MODEL_SAVE_PATH' not in globals():
    print("MODEL_SAVE_PATH not found. Please run the configuration loading cell first.")
    model_path = './best_pneumonia_cnn.keras' # Fallback
else:
    model_path = MODEL_SAVE_PATH

# Define custom objects, especially for metrics used during compilation
custom_objects = {
    'precision': Precision(),
    'recall': Recall(),
    'auc': AUC()
}

try:
    loaded_model = tf.keras.models.load_model(model_path, custom_objects=custom_objects)
    print("Model loaded successfully!")
    loaded_model.summary()
except Exception as e:
    print(f"Error loading model from {model_path}: {e}")
    print("Please ensure the model exists at the specified path and custom objects are correctly defined.")

In [ ]:
print("Plotting training history...")
classifier.plot_training_history()

In [ ]:
print("Evaluating on test set...")
test_metrics = classifier.evaluate(test_gen)

print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"\nTest Accuracy: {test_metrics['accuracy']*100:.2f}%")
print(f"Test Precision: {test_metrics['precision']*100:.2f}%")
print(f"Test Recall: {test_metrics['recall']*100:.2f}%")
print(f"Test AUC: {test_metrics['auc']:.4f}")

In [ ]:
print("Visualizing predictions...")
classifier.predict_and_visualize(test_gen, num_samples=9)

In [ ]:
# Download the trained model to your computer
from google.colab import files
import os

# Use MODEL_SAVE_PATH from the loaded configuration
# Ensure the configuration is loaded from a previous cell first
if 'MODEL_SAVE_PATH' not in globals():
    print("MODEL_SAVE_PATH not found. Please run the configuration loading cell first.")
    model_path = './best_pneumonia_cnn.keras' # Update to your path
else:
    model_path = MODEL_SAVE_PATH

if os.path.exists(model_path):
    try:
        files.download(model_path)
        print("✓ Model downloaded! You can now use it offline.")
    except Exception as e:
        print(f"Error during download: {e}")
        print("Ensure your browser allows downloads from Colab.")
else:
    print(f"Model file not found at {model_path}!")
    print("Please ensure the training was successful and the model was saved.")